
**TIME SERIES AND ECONOMETRICS**
## Modeling non-stationarity and finding an equilibrium: COINTEGRATION ANALYSIS OF XOM AND CVX

### Student Information
**Prepared by:** [Joannah G Mawutsa]  
**Registration Number:** [R2424533]

Program : [Data science and Systems]  
**Course:** Time Series and Econometrics  

This project uses cointegration analysis together with an Error Correction Model (ECM) to study non-stationary behavior and determine long-run equilibrium relationships between two financial time series. Daily closing stock prices for ExxonMobil (XOM) and Chevron (CVX) covering the period from February 2018 to December 2025 were analyzed. The study involved testing for unit roots, assessing cointegration, estimating the ECM, calibrating parameters, and evaluating the overall model performance.

The findings show that both time series are individually non-stationary; however, they demonstrate significant cointegration (p = 0.0430), confirming the existence of a strong long-run equilibrium relationship.

Dataset Selection

Among the stock pairs examined, the XOM + CVX combination was chosen as the most suitable based on the following factors:

Both series are non-stationary, meeting the primary requirement for cointegration analysis.
The pair shows significant cointegration (p < 0.05), indicating a statistically valid equilibrium relationship.
ExxonMobil and Chevron operate within the same macroeconomic environment in the Energy Sector and are similarly influenced by global crude oil price movements, making their equilibrium relationship economically meaningful and dependable.
The dataset spans a sufficiently long period (2018–2025, ~1989 observations), enabling reliable estimation of the ECM.

### 1. Definition & Description

To analyze non-stationary data and determine the existence of a long-run equilibrium relationship, the Engle–Granger Two-Step Cointegration approach together with an Error Correction Model (ECM) is employed.
**Equation 1: The Cointegrating Regression (The Long-Run Equilibrium)**


$$Y_t = \beta_0 + \beta_1 X_t + e_t$$

* **$Y_t$**: The non-stationary log price of the target asset (XOM) at time $t$.
* **$X_t$**: The non-stationary log price of the benchmark asset (CVX) at time $t$.
* **$\beta_0$**: The intercept.
* **$\beta_1$**: The cointegrating coefficient (hedge ratio).
* **$e_t$**: The residual error term at time $t$. For equilibrium to exist, this error term must be stationary $I(0)$.

**Equation 2: The Error Correction Model (The Short-Run Dynamics)**


$$\Delta Y_t = \alpha + \gamma e_{t-1} + \sum_{i=1}^{p} \phi_i \Delta Y_{t-i} + \sum_{j=1}^{q} \theta_j \Delta X_{t-j} + \epsilon_t$$

* **$\Delta Y_t$ & $\Delta X_t$**: The first differences (returns) of the assets, which are stationary.
* **$e_{t-1}$**: The lagged residual from Equation 1, representing the error or deviation from long-run equilibrium in the previous period.
* **$\gamma$**: The Error Correction Term (ECT) or speed of adjustment. It measures how quickly the system corrects itself back to equilibrium. (It must be negative and statistically significant).
* **$\epsilon_t$**: White noise error.

**Description:**
Cointegration exists when two non-stationary time series move together over time due to a shared stochastic trend, such that a linear combination of the series becomes stationary and mean-reverting. The Error Correction Model represents this relationship by combining the long-run equilibrium relationship between the variables with the short-run adjustment process that corrects deviations from equilibrium.








**2. Demonstration**

**2.1 Data Acquisition and Structuring**



In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.graphics.tsaplots import plot_acf
import scipy.stats as stats
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

#importing data
tickers = ['XOM', 'CVX']
data = yf.download(tickers, start='2018-02-01', end='2025-12-31', progress=False)['Close']

# Log Transformation
log_prices = np.log(data).dropna()
print("Data Acquisition Complete. Observations:", len(log_prices))
log_prices.tail()

**2.2 Stationarity Testing (ADF)**:


In [ ]:

for stock in ['XOM', 'CVX']:
    res = adfuller(log_prices[stock])
    print(f"{stock} ADF Stat: {res[0]:.4f} | p-value: {res[1]:.4f}")



**Interpretation:**  XOM (p=0.8685) and CVX (p=0.5761) both have p-values significantly greater than 0.05.
* We fail to reject the null hypothesis of a unit root. This confirms both series are Non-Stationary, fulfilling the first requirement for cointegration.



**2.3 Cointegration Testing**:
We use the Engle-Granger test to determine if a stationary linear combination exists between these two Non-Stationary series.

In [ ]:
score, pval, _ = coint(log_prices['XOM'], log_prices['CVX'])
print(f"Cointegration Test Statistic: {score:.4f}")
print(f"Cointegration p-value: {pval:.4f}")

**Interpretation:** The p-value of 0.0430 is below the 0.05 threshold. We reject the null hypothesis of no cointegration. This statistically proves that XOM and CVX share a long-run equilibrium, meaning they do not drift apart indefinitely.

### 2.4: Error Correction Model (ECM) Estimation
We estimate the short-run dynamics and the speed of adjustment toward the equilibrium.

In [ ]:
X_long = sm.add_constant(log_prices['CVX'])
long_run_model = sm.OLS(log_prices['XOM'], X_long).fit()
beta_1 = long_run_model.params['CVX']
residuals = long_run_model.resid

dy = log_prices['XOM'].diff().dropna()
dx = log_prices['CVX'].diff().dropna()
ect = residuals.shift(1).dropna()

X_short = sm.add_constant(pd.concat([ect, dx], axis=1))
X_short.columns = ['Intercept', 'ECT_Speed_of_Adj', 'Short_Run_CVX']
ecm_model = sm.OLS(dy.loc[ect.index], X_short).fit()

gamma = ecm_model.params['ECT_Speed_of_Adj']
theta = ecm_model.params['Short_Run_CVX']


print(f"Hedge Ratio (Beta 1): {beta_1:.4f}")
print("\n--- ERROR CORRECTION MODEL (PHASE 3) ---")
print(ecm_model.summary().tables[1])

print("\n" + "="*40)
print(f"{'CALIBRATED PARAMETER':<25} | {'VALUE':<10}")
print("-" * 40)
print(f"{'Beta 1 (Hedge Ratio)':<25} | {beta_1:.4f}")
print(f"{'Gamma (Speed of Adj)':<25} | {gamma:.4f}")
print(f"{'Theta (Short-Run Impact)':<25} | {theta:.4f}")
print("="*40)

**Interpretation of Calibrated Parameters:**
1.  **ECT (Speed of Adjustment) = -0.0079**: The negative sign confirms the stability of the equilibrium. It indicates that 0.79% of any price divergence is corrected daily.
2.  **Short_Run_CVX = 0.8083**: In the short term, a 1% change in CVX results in a 0.81% immediate change in XOM.
3.  **Statistical Significance**: Both coefficients have p-values < 0.05, confirming the model's validity in explaining the price dynamics.

## 3. Diagram (Exploratory Plots)
The diagrams below demonstrate the movement from non-stationary price series toward a stationary long-run equilibrium relationship.
Price Co-movement: Illustrates the upward stochastic trends followed by XOM and CVX over time.
Equilibrium Spread: Displays the stationary residual series obtained from the cointegrating regression, representing the long-run equilibrium anchor of the system.



In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(log_prices['XOM'], label='XOM Log Price', color='navy', linewidth=1.5)
ax1.plot(log_prices['CVX'], label='CVX Log Price', color='darkorange', linewidth=1.5)
ax1.set_title('Exploratory Plot 1: Price Co-movement (Non-Stationary Levels)', fontsize=14)
ax1.set_ylabel('Log Price')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(residuals, label='Residual Spread (Equilibrium)', color='purple', linewidth=1.2)
ax2.axhline(0, color='black', linestyle='--', label='Mean (Zero)')
ax2.set_title('Exploratory Plot 2: Mean-Reverting Spread (Stationary Equilibrium)', fontsize=14)
ax2.set_ylabel('Deviation')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation of Diagrams:**
* Plot 1 visually verifies the presence of non-stationarity, as both asset prices display an upward trend over time, indicating that they move with a shared stochastic pattern.
Plot 2 presents the transformed data. Using the estimated hedge ratio, the benchmark asset (CVX) is subtracted from the target asset (XOM) to produce a stationary spread. The spread’s repeated movement back toward the zero line serves as visual evidence of a stable equilibrium relationship.



## 4. Diagnosis & Damage
This section examines the statistical reliability of the model. Diagnostic plots are used to visualize the residual behavior and evaluate the model’s performance in relation to the key challenges associated with time series analysis.
### 4.1 Diagnostic Plots
The following plots enable us to examine the residuals for randomness, distributional properties, and underlying patterns.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals over time (Checking for Homoscedasticity & Structural Breaks)
ax1.plot(ecm_model.resid, color='teal', linewidth=0.8)
ax1.axhline(0, color='red', linestyle='--')
ax1.set_title('Residuals over Time')

# 2. Histogram (Checking for Normality)
ax2.hist(ecm_model.resid, bins=50, color='gray', edgecolor='black', alpha=0.7)
ax2.set_title('Histogram of Residuals')

# 3. Q-Q Plot (Checking for Fat Tails)
sm.qqplot(ecm_model.resid, line='45', ax=ax3)
ax3.set_title('Normal Q-Q Plot')

# 4. Autocorrelation Plot (Checking for Independence)
plot_acf(ecm_model.resid, ax=ax4, lags=40)
ax4.set_title('Residual Autocorrelation (ACF)')

plt.tight_layout()
plt.show()

### 4.2 Identification of Model Damage and Challenges
Diagnostic Performance Summary
Non-Stationarity (Resolved): The model successfully stabilizes the data. Although the initial inputs were non-stationary, the transformation into a stationary equilibrium is confirmed by the mean-reverting behavior observed in the residual plots.

Autocorrelation (Pass): The model accurately accounts for temporal dependencies. Since the ACF plot shows residuals largely within the confidence intervals, we can conclude that the time-related patterns have been effectively captured.

Non-Normality (Weakness Identified): The model struggles with extreme values. Histogram and Q-Q plots indicate leptokurtosis (fat tails), a common byproduct of financial shocks—like the 2020 oil crash—that linear frameworks fail to fully absorb.

Heteroscedasticity (Potential Issue): Consistency of variance is a concern. Volatility clustering between 2020 and 2022 suggests that while the model finds the mean equilibrium, the "noise" or risk level fluctuates over time.

Structural Breaks (Weakness Identified): There is evidence of regime instability. Significant residual spikes during the COVID-19 era suggest that the energy market underwent a fundamental shift that a fixed linear Error Correction Model (ECM) may not fully accommodate.

Multicollinearity (Managed): While XOM and CVX are naturally correlated, the model’s design mitigates risk. By focusing on first differences within the ECM, the analysis avoids the "spurious regression" traps typically found in standard OLS models.


Final Quality Assessment
The model is highly reliable for forecasting direction and equilibrium adjustments, supported by a significant Error Correction Term (ECT) and a clean ACF. However, due to the identified issues with Normality and Heteroscedasticity, the model likely underestimates the probability and impact of extreme market outliers.


### 5. Directions for Model Improvement

Shortening the Time Horizon: The "Residuals over Time" plot highlights extreme volatility during the 2020 global energy crisis. We should evaluate re-estimating the model using a post-2021 subsample. Eliminating this "black swan" period would likely normalize the residuals and produce parameter estimates that are more representative of current market conditions.

Outlier Mitigation: To address the failed Jarque-Bera test caused by radical daily price swings, we could implement Winsorization. By capping extreme values at the 1st and 99th percentiles, we prevent isolated outliers from distorting the long-run hedge ratio.

Volatility Weighting: To resolve the detected heteroscedasticity, we can apply a GARCH-based weighting to the returns. This approach de-emphasizes high-volatility periods, effectively reducing noise within the Error Correction Term.

Decision: We advise maintaining the current model as the baseline for long-term strategic planning. However, for high-frequency trading applications, we recommend shortening the horizon to the most recent 3 years of data to achieve a tighter, more statistically robust fit.




### 6. Deployment

The intended application for this Cointegration/ECM model is integration into an Automated Pairs Trading Framework. The following outline details the operational workflow required to utilize this model within a live production environment:
#### 6.1 Signal Generation (The Long-Run Strategy)

The model will be deployed to monitor the Spread or residual term ($e_t$) in real-time.

* **Deployment Thresholds:** We define entry signals based on Standard Deviations ($\sigma$) from the equilibrium mean (zero).
* **Execution:** When the spread exceeds $+2\sigma$ (meaning XOM is significantly overvalued relative to CVX), the system will automatically trigger a Sell XOM / Buy CVX order.
* **Closing the Position:** The model will trigger a Close signal when the spread returns to $\pm 0.5\sigma$, capturing the profit from the mean-reversion predicted by our Speed of Adjustment ($\gamma$).

#### 6.2 Risk Management and Monitoring

Implementation of the model necessitates a circuit breaker mechanism specifically engineered to address the vulnerabilities identified in our Diagnosis & Damage assessment.

* **Stop-Loss:** If the spread continues to diverge beyond $4\sigma$, it indicates a possible Structural Break (one of the 6 challenges). The system must automatically liquidate positions, as the cointegration relationship may have fundamentally collapsed.
* **Hedge Ratio Maintenance:** The Hedge Ratio ($\beta_1 = 1.3463$) should be recalibrated on a rolling 60-day window to ensure the position size remains market-neutral as the energy sector evolves.

#### 6.3 Technical Infrastructure

* Environment: The model will be hosted on a cloud-based server (e.g., Google Cloud or AWS) running a Python-based execution engine.

Data Pipeline: A live API connection to a financial data provider (like Bloomberg or Interactive Brokers) will feed the model high-frequency intraday prices.
* **Latency Management:** Because our Speed of Adjustment ($\gamma = -0.0079$) is relatively slow, this model is best deployed as a Swing Trading strategy (holding positions for days/weeks) rather than a high-frequency strategy, reducing the need for expensive low-latency hardware.






## 7. Bibliography

Engle, Robert F., and Clive W. J. Granger. Co-Integration and Error Correction: Representation, Estimation, and Testing. *Econometrica*, vol. 55, no. 2, 1987, pp. 251-276.

Johansen, Søren. Estimation and Hypothesis Testing of Cointegration Vectors in Gaussian Vector Autoregressive Models. *Econometrica*, vol. 59, no. 6, 1991, pp. 1551-1580.

Tsay, Ruey S. *Analysis of Financial Time Series*. 3rd ed., Wiley, 2010.

Yahoo Finance. Chevron Corporation (CVX) Historical Prices. *Yahoo!*, 2026, finance.yahoo.com/quote/CVX/history/.

Yahoo Finance. Exxon Mobil Corporation (XOM) Historical Prices. *Yahoo!*, 2026, finance.yahoo.com/quote/XOM/history/.